In [ ]:
from pinecone import Pinecone
import json
import os
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import torch
from collections import defaultdict
from collections.abc import Sequence


# Municipality-RAG Indexing Notebook

# For articles - embed the title + subtitle if subtitle is not None
# For services - embed the title + description
# Divide data into namespaces:
# namespaces: haifa, tel-aviv, municipal-services, city-planning, waste-management,
# transportation, education, culture-recreation, social-services, general


In [ ]:
CATEGORY_TO_NAMESPACE = {
    "haifa": "haifa",
    "tel-aviv": "tel-aviv",
    "municipal-services": "municipal-services",
    "services": "municipal-services",
    "city-planning": "city-planning",
    "planning": "city-planning",
    "construction": "city-planning",
    "waste-management": "waste-management",
    "waste": "waste-management",
    "recycling": "waste-management",
    "transportation": "transportation",
    "parking": "transportation",
    "traffic": "transportation",
    "education": "education",
    "schools": "education",
    "culture-recreation": "culture-recreation",
    "culture": "culture-recreation",
    "recreation": "culture-recreation",
    "social-services": "social-services",
    "social": "social-services",
    "general": "general"
}


### Setup


In [ ]:
load_dotenv()
# Keys and Paths
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX = os.getenv("PINECONE_INDEX")
PINECONE_ENVIRONMENT = os.getenv("PINECONE_ENVIRONMENT")
ARTICLES_PATH = "data/article_meta_data.json"
SERVICES_PATH = "data/service_meta_data.json"
ANNOUNCEMENTS_PATH = "data/announcement_meta_data.json"


In [ ]:
pc = Pinecone(api_key=PINECONE_API_KEY, environment=PINECONE_ENVIRONMENT)
index = pc.Index(PINECONE_INDEX)


### Loading data


In [ ]:
def read_json(path: str) -> json:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


In [ ]:
# Load data files (create empty lists if files don't exist)
articles = read_json(ARTICLES_PATH) if os.path.exists(ARTICLES_PATH) else []
services = read_json(SERVICES_PATH) if os.path.exists(SERVICES_PATH) else []
announcements = read_json(ANNOUNCEMENTS_PATH) if os.path.exists(ANNOUNCEMENTS_PATH) else []


In [ ]:
def safe_strip(val):
    # Returns a stripped string or empty string if None/empty/not str
    if isinstance(val, str):
        return val.strip()
    if val is None:
        return ""
    return str(val).strip()

def deduplicate_items(items, key_fields):
    """Deduplicate items based on key fields."""
    seen = set()
    unique_items = []
    for item in items:
        sig = tuple(safe_strip(item.get(key)) for key in key_fields)
        if sig not in seen:
            unique_items.append(item)
            seen.add(sig)
    return unique_items

articles = deduplicate_items(articles, ["title", "subtitle", "article_text"])
services = deduplicate_items(services, ["title", "description"])
announcements = deduplicate_items(announcements, ["title", "content"])

print(f"{len(articles)} unique articles remain after deduplication.")
print(f"{len(services)} unique services remain after deduplication.")
print(f"{len(announcements)} unique announcements remain after deduplication.")


### Loading Embedding Model


In [ ]:
# Embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = SentenceTransformer("intfloat/multilingual-e5-large", device=device)


### Preparing Data before Embeddings


In [ ]:
# Creating the fields we want in our embeddings of articles
for article in articles:
    # If subtitle exists and is not empty/None, include it. Otherwise, use only title.
    subtitle = article.get("subtitle")
    if subtitle and subtitle.strip():
        article["to_embed"] = f"{article['title']}\n{subtitle.strip()}"
        article['content'] = f"{article['title']}\n{subtitle.strip()}\n{article.get('article_text', '')}\n links: {article.get('page_link', '')}"
    else:
        article["to_embed"] = article.get("title", "")
        article['content'] = f"{article.get('title', '')}\n{article.get('article_text', '')}\n **links:** {article.get('page_link', '')}"


In [ ]:
# Creating the fields we want in our embeddings of services
for service in services:
    title = service.get("title", "")
    description = service.get("description", "")
    service["to_embed"] = f"{title}\n{description}"
    service['content'] = f"{title}\n{description}\n" + "\n".join(service.get("details", [])) + f"\n links: {service.get('page_link', '')}"


In [ ]:
# Creating the fields we want in our embeddings of announcements
for announcement in announcements:
    title = announcement.get("title", "")
    content = announcement.get("content", "")
    announcement["to_embed"] = f"{title}\n{content[:200]}"  # First 200 chars of content
    announcement['content'] = f"{title}\n{announcement.get('date', '')}\n{content}\n links: {announcement.get('page_link', '')}"


In [ ]:
def assign_namespace(categories, title, page_link):
    """
    Assign a namespace based on category names and URL.
    - Uses substring matching so 'city-planning' maps to 'city-planning', etc.
    - Checks URL for city name (haifa, tel-aviv)
    """
    # Check URL for city
    if page_link:
        page_link_lower = page_link.lower()
        if "haifa" in page_link_lower or "חיפה" in page_link_lower:
            return "haifa"
        elif "tel-aviv" in page_link_lower or "telaviv" in page_link_lower or "תל-אביב" in page_link_lower:
            return "tel-aviv"
    
    # Check title for city
    if title:
        title_lower = title.lower()
        if "חיפה" in title or "haifa" in title_lower:
            return "haifa"
        elif "תל אביב" in title or "tel aviv" in title_lower:
            return "tel-aviv"
    
    # Check categories
    for cat in categories:
        cat_lower = cat.lower()
        for ns_key in CATEGORY_TO_NAMESPACE:
            if ns_key in cat_lower:
                return CATEGORY_TO_NAMESPACE[ns_key]
    
    return "general"  # default

# For articles
for article in articles:
    categories = article.get("categories", [])
    page_link = article.get("page_link", "")
    title = article.get("title", "")
    article["namespace"] = assign_namespace(categories, title, page_link)

# For services
for service in services:
    categories = service.get("categories", [])
    page_link = service.get("page_link", "")
    title = service.get("title", "")
    service["namespace"] = assign_namespace(categories, title, page_link)

# For announcements
for announcement in announcements:
    categories = announcement.get("categories", [])
    page_link = announcement.get("page_link", "")
    title = announcement.get("title", "")
    announcement["namespace"] = assign_namespace(categories, title, page_link)


In [ ]:
# Flatten services and announcements into document format
service_docs = []
for idx, service in enumerate(services):
    service_docs.append({
        "id": f"service_{idx}",
        "to_embed": service["to_embed"],
        "namespace": service["namespace"],
        "page_link": service.get("page_link", ""),
        "categories": service.get("categories", []),
        "title": service.get("title", ""),
        "description": service.get("description", ""),
        "details": service.get("details", []),
        "content": service.get("content", "")
    })

announcement_docs = []
for idx, announcement in enumerate(announcements):
    announcement_docs.append({
        "id": f"announcement_{idx}",
        "to_embed": announcement["to_embed"],
        "namespace": announcement["namespace"],
        "page_link": announcement.get("page_link", ""),
        "categories": announcement.get("categories", []),
        "title": announcement.get("title", ""),
        "date": announcement.get("date", ""),
        "content": announcement.get("content", "")
    })

# Add IDs to articles if missing
for idx, article in enumerate(articles):
    if "id" not in article:
        article["id"] = f"article_{idx}"


### Uploading data


In [ ]:
def safe_str(x):
    """Convert None or non-string to empty string. Leave string as is."""
    return x if isinstance(x, str) and x is not None else ""

def prepare_embeddings(docs, embed_model, type_):
    """
    Calculate embeddings and package metadata for Pinecone upsert.
    Returns: Dict[namespace, List[(id, embedding, metadata)]]
    """
    namespace_batches = defaultdict(list)
    texts = [doc["to_embed"] for doc in docs]
    embeddings = embed_model.encode(
        ["passage: " + text for text in texts],
        batch_size=32, show_progress_bar=True
    )
    for doc, embedding in zip(docs, embeddings):
        metadata = {
            "type": type_,
            "page_link": safe_str(doc.get("page_link")),
            "categories": doc.get("categories", []),
            "id": doc.get("id", ""), 
        }
        if type_ == "article":
            metadata.update({
                "title": safe_str(doc.get("title", "")),
                "subtitle": safe_str(doc.get("subtitle", "")),
                "image_links": doc.get("image_links", []),
                "content": safe_str(doc.get("content", "")), 
            })
        elif type_ == "service":
            metadata.update({
                "title": safe_str(doc.get("title", "")),
                "description": safe_str(doc.get("description", "")),
                "details": doc.get("details", []),
                "content": safe_str(doc.get("content", "")),
            })
        elif type_ == "announcement":
            metadata.update({
                "title": safe_str(doc.get("title", "")),
                "date": safe_str(doc.get("date", "")),
                "content": safe_str(doc.get("content", "")),
            })
        ns = doc["namespace"]
        namespace_batches[ns].append((doc["id"], embedding.tolist(), metadata))
    return namespace_batches


In [ ]:
article_batches = prepare_embeddings(articles, embed_model, "article")
service_batches = prepare_embeddings(service_docs, embed_model, "service")
announcement_batches = prepare_embeddings(announcement_docs, embed_model, "announcement")


In [ ]:
BATCH_SIZE = 16

def clear_all_namespaces(index):
    """Clear all existing namespaces from the index."""
    ns_to_delete = list(index.describe_index_stats()["namespaces"].keys())
    for ns in ns_to_delete:
        index.delete(delete_all=True, namespace=ns)
    print(f"Deleted namespaces: {ns_to_delete}")

# Uncomment to clear all namespaces before uploading
# clear_all_namespaces(index)

def upsert_to_pinecone(namespace_batches):
    """Upsert documents to Pinecone by namespace."""
    for namespace, records in namespace_batches.items():
        print(f"Uploading {len(records)} documents to namespace '{namespace}'...")
        for i in range(0, len(records), BATCH_SIZE):
            batch = records[i:i+BATCH_SIZE]
            index.upsert(vectors=batch, namespace=namespace)
        print(f"Completed namespace '{namespace}'")

upsert_to_pinecone(article_batches)
upsert_to_pinecone(service_batches)
upsert_to_pinecone(announcement_batches)


In [ ]:
stats = index.describe_index_stats()
print(stats)
